# 第一章：环境搭建与核心概念

## 学习目标
- 使用 uv 搭建 LangGraph 开发环境
- 理解 LangGraph 与 LangChain 的关系和区别
- 掌握核心概念：StateGraph、节点（Node）、边（Edge）
- 构建并运行第一个状态图
- 学会可视化图结构

## 1. 环境搭建（uv）

本项目使用 **uv** 管理 Python 环境和依赖。

```bash
# 在项目目录下初始化（首次）
uv init --python 3.11
uv add langgraph langchain-core langchain-openai

# 启动 Jupyter Lab
uv run --with jupyter jupyter lab
```

若已在 uv 环境中运行此 Notebook，直接执行下方单元格验证安装即可。

In [ ]:
# 验证安装 & 版本确认
from importlib.metadata import version

print(f"langgraph 版本: {version('langgraph')}")
print(f"langchain-core 版本: {version('langchain-core')}")

## 2. 配置 API Key

本教程通过 OpenAI 兼容接口接入模型。Key 写入项目根目录的 `.env` 文件或 shell 配置文件：

```bash
echo 'export Qwen_API_KEY="sk-..."' >> ~/.bashrc && source ~/.bashrc
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

if os.environ.get("Qwen_API_KEY"):
    print("✓ 已找到环境变量 Qwen_API_KEY")
else:
    print("✗ 未找到环境变量 Qwen_API_KEY，请先设置")

In [ ]:
from langchain_openai import ChatOpenAI

# 初始化模型（后续各章统一使用此配置）
llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

## 3. 为什么需要 LangGraph？

如果你学过 LangChain，已经知道如何用 LCEL（`|` 管道）把组件串成链。但 LCEL 本质上是**线性管道**——数据从左到右流过，不能回头。

现实中很多任务需要**循环和分支**：
- Agent 调用工具后，要根据结果决定下一步（继续调用？还是返回？）
- 多轮审批流程，可能被打回重做
- 多个 Agent 协作，轮流执行

LangGraph 用**有向图**解决这个问题：节点是处理步骤，边是流转方向，可以有条件分支和循环。

| | LangChain LCEL | LangGraph |
|--|--|--|
| 结构 | 线性管道 | 有向图（支持循环） |
| 适合场景 | 简单 prompt → model → parser | 多步骤、有条件分支的 Agent |
| 状态管理 | 数据沿管道传递 | 显式定义 State，所有节点共享 |
| 关系 | 基础框架 | 建立在 langchain-core 之上 |

## 4. 第一个 LangGraph 程序

先看代码，再解释。我们构建一个最简单的图：接收用户问题 → 调用模型 → 返回结果。

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 第一步：定义状态（State）
class State(TypedDict):
    question: str
    answer: str

# 第二步：定义节点函数（接收 state，返回更新）
def ask_model(state: State) -> dict:
    response = llm.invoke(state["question"])
    return {"answer": response.content}

# 第三步：构建图
graph = StateGraph(State)
graph.add_node("ask_model", ask_model)  # 添加节点
graph.add_edge(START, "ask_model")      # 起点 → 节点
graph.add_edge("ask_model", END)        # 节点 → 终点

# 第四步：编译并运行
app = graph.compile()
result = app.invoke({"question": "用一句话解释什么是状态机", "answer": ""})
print(result["answer"])

### 刚才发生了什么？

整个流程分四步，这也是 LangGraph 的固定套路：

1. **定义 State**：用 `TypedDict` 声明图中所有节点共享的数据结构。这里有两个字段：`question`（输入）和 `answer`（输出）。
2. **定义节点函数**：普通 Python 函数，接收当前 `state`，返回一个 dict 表示要更新哪些字段。注意：不需要返回完整 state，只返回变化的部分。
3. **构建图**：创建 `StateGraph`，添加节点和边。`START` 和 `END` 是特殊常量，表示图的入口和出口。
4. **编译并运行**：`.compile()` 产出一个可执行对象（`CompiledGraph`），它和 LangChain 的 Runnable 一样支持 `.invoke()` / `.stream()` / `.batch()`。

### 常见问题

- **忘记 `add_edge(START, ...)`**：图没有入口，编译时会报错。
- **忘记 `add_edge(..., END)`**：图没有出口，运行时会卡住。
- **节点函数返回了完整 state**：不会报错，但不推荐。只返回要更新的字段，LangGraph 会自动合并。

## 5. 对比：不用 LangGraph vs 用 LangGraph

上面的例子只有一个节点，看起来比直接调用模型还复杂。先看「不用 LangGraph」的写法：

In [ ]:
# ❌ 不用 LangGraph：直接调用
response = llm.invoke("用一句话解释什么是状态机")
print(response.content)

In [ ]:
# ✅ 用 LangGraph：结构化的图
result = app.invoke({"question": "用一句话解释什么是状态机", "answer": ""})
print(result["answer"])

单节点时 LangGraph 确实更繁琐。但它的价值在后续章节中体现：

- **条件分支**：根据模型输出走不同路径（第三章）
- **循环**：Agent 反复调用工具直到任务完成（第四章）
- **人工干预**：在关键步骤暂停等待人类审批（第五章）
- **状态持久化**：保存和恢复图的执行状态（第七章）

这些都依赖图结构和显式 State，普通函数调用做不到。

## 6. 可视化图结构

LangGraph 内置了 Mermaid 图表生成，方便理解图的拓扑结构。

In [ ]:
# 输出 Mermaid 格式的图描述
print(app.get_graph().draw_mermaid())

将上面的文本粘贴到 [Mermaid Live Editor](https://mermaid.live) 可以看到可视化结果。也可以直接在 Jupyter 中渲染：

In [ ]:
from IPython.display import display, Markdown

# 在 Jupyter 中渲染 Mermaid 图
mermaid_code = app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

> **提示**：Mermaid 渲染需要 JupyterLab 支持。如果看不到图，直接看文本输出即可，不影响学习。

## 7. 多节点图

接下来构建一个更实际的例子：先让模型回答问题，再让模型翻译成英文。两个节点串联。

In [ ]:
# 定义状态：增加翻译字段
class TranslateState(TypedDict):
    question: str
    answer_zh: str
    answer_en: str

# 节点 1：中文回答
def answer_in_chinese(state: TranslateState) -> dict:
    response = llm.invoke(f"用中文简短回答：{state['question']}")
    return {"answer_zh": response.content}

# 节点 2：翻译成英文
def translate_to_english(state: TranslateState) -> dict:
    response = llm.invoke(f"将以下内容翻译成英文，只输出翻译结果：\n{state['answer_zh']}")
    return {"answer_en": response.content}

# 构建图
graph2 = StateGraph(TranslateState)
graph2.add_node("answer", answer_in_chinese)
graph2.add_node("translate", translate_to_english)
graph2.add_edge(START, "answer")
graph2.add_edge("answer", "translate")
graph2.add_edge("translate", END)

app2 = graph2.compile()

In [ ]:
# 运行多节点图
result = app2.invoke({
    "question": "Python 的 GIL 是什么",
    "answer_zh": "",
    "answer_en": "",
})

print("中文回答：")
print(result["answer_zh"])
print("\n英文翻译：")
print(result["answer_en"])

In [ ]:
# 查看图结构
print(app2.get_graph().draw_mermaid())

### 刚才发生了什么？

1. State 定义了三个字段，两个节点各负责填充一个。
2. `answer` 节点读取 `question`，写入 `answer_zh`。
3. `translate` 节点读取 `answer_zh`，写入 `answer_en`。
4. 数据通过 State 在节点间传递——每个节点只关心自己需要的字段。

这就是 LangGraph 的核心思路：**State 是数据中心，节点是处理单元，边决定执行顺序**。

## 8. 流式输出

编译后的图同样支持 `.stream()`，但行为与 LangChain 不同：它按**节点**逐步输出，每个节点完成后 yield 一次结果。

In [ ]:
# 流式输出：按节点逐步返回
for step in app2.stream({
    "question": "什么是递归",
    "answer_zh": "",
    "answer_en": "",
}):
    # step 是 {节点名: 该节点返回的更新}
    for node_name, update in step.items():
        print(f"--- 节点 [{node_name}] 完成 ---")
        for key, value in update.items():
            print(f"  {key}: {value[:80]}..." if len(str(value)) > 80 else f"  {key}: {value}")
        print()

### 刚才发生了什么？

`.stream()` 每执行完一个节点就 yield 一次，格式是 `{节点名: 更新内容}`。这对调试很有用——可以看到每个节点做了什么。

| 方法 | 输出粒度 | 适用场景 |
|------|---------|----------|
| `.invoke()` | 全部执行完返回最终 state | 简单调用 |
| `.stream()` | 每个节点完成后 yield | 调试、进度展示 |

## 9. 核心概念总览

| 概念 | 说明 |
|------|------|
| **State** | 用 `TypedDict` 定义的共享数据结构，所有节点都可以读写 |
| **Node（节点）** | 普通 Python 函数，接收 state，返回要更新的字段 |
| **Edge（边）** | 连接节点的有向边，决定执行顺序 |
| **START / END** | 特殊常量，表示图的入口和出口 |
| **StateGraph** | 图的构建器，用于添加节点和边 |
| **compile()** | 将图编译为可执行对象，支持 invoke / stream / batch |

用一句话概括 LangGraph 的编程模型：

> **定义 State → 编写节点函数 → 用边连接节点 → 编译运行**

## 总结

本章学习了：
- ✅ 使用 uv 搭建 LangGraph 开发环境
- ✅ LangGraph 与 LangChain 的关系：图结构 vs 线性管道
- ✅ 核心四步：定义 State → 编写节点 → 连接边 → 编译运行
- ✅ `StateGraph`、`START`、`END` 的基本用法
- ✅ 多节点图的构建和数据流转
- ✅ `.invoke()` 和 `.stream()` 两种执行方式
- ✅ 用 Mermaid 可视化图结构

## 下一章预告

**第二章：State 设计（TypedDict、Annotated reducer）** —— 本章的 State 只能「覆盖」字段。下一章学习如何用 reducer 实现「追加」等更灵活的更新策略，这是构建多轮对话和复杂 Agent 的基础。